In [3]:
%%bash

mkdir ruGPT-3.5-training
cd ruGPT-3.5-training

mkdir: cannot create directory ‘ruGPT-3.5-training’: File exists


In [6]:
%pip install llm-rs==0.2.15

Defaulting to user installation because normal site-packages is not writeable
DEPRECATION: distro-info 0.23ubuntu1 has a non-standard version number. pip 23.3 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of distro-info or contact the author to suggest that they release a version with a conforming version number. Discussion can be found at https://github.com/pypa/pip/issues/12063
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install numpy>=1.26.0
%pip install scipy>=1.11.0
%pip install torch==2.0.1
%pip install accelerate==0.23.0
%pip install bitsandbytes==0.41.1
%pip install peft==0.5.0
%pip install transformers==4.34.0
%pip install pillow==10.0.1
%pip install llama-cpp-python
%pip install datasets
%pip install zstandard
%pip install jsonlines
%pip install wandb
%pip install openai
%pip install sentencepiece
%pip install fire
%pip install datasketch==1.5.9
%pip install nltk==3.8.1
%pip install scikit-learn==1.3.0
%pip install pytest
%pip install llm-rs==0.2.15
%pip install tqdm>=4.66.1

DEPRECATION: distro-info 0.23ubuntu1 has a non-standard version number. pip 23.3 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of distro-info or contact the author to suggest that they release a version with a conforming version number. Discussion can be found at https://github.com/pypa/pip/issues/12063
Note: you may need to restart the kernel to use updated packages.
DEPRECATION: distro-info 0.23ubuntu1 has a non-standard version number. pip 23.3 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of distro-info or contact the author to suggest that they release a version with a conforming version number. Discussion can be found at https://github.com/pypa/pip/issues/12063
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
DEPRECATION: distro-info 0.23ubuntu1 has a non-standard version number. pip 23.3 will en

In [5]:
!git clone https://github.com/IlyaGusev/rulm.git

fatal: destination path 'rulm' already exists and is not an empty directory.


In [1]:
%%bash

mkdir {configs,internal_prompts,output,output_ggml}
cp rulm/self_instruct/configs/gigasaiga_13b.json configs/rugpt35_13b.json
cp rulm/self_instruct/internal_prompts/gigasaiga.json internal_prompts/rugpt35.json

mkdir: cannot create directory ‘configs’: File exists
mkdir: cannot create directory ‘internal_prompts’: File exists
mkdir: cannot create directory ‘output’: File exists


In [1]:
import subprocess
from pathlib import Path

In [ ]:
# Set up paths
content_dir = Path('.').resolve()
train_full_path = content_dir / 'train_full.jsonl'
val_full_path = content_dir / 'val_full.jsonl'

# Run create_chat_set script from rulm
module_directory = Path('rulm/self_instruct').resolve()
subprocess.run(
    ['python3', '-m', 'src.data_processing.create_chat_set', str(train_full_path), str(val_full_path)],
    cwd=module_directory,
    check=True
)

# Check if train_full.jsonl exists
if not train_full_path.exists():
    raise FileNotFoundError(f"{train_full_path} does not exist")

In [ ]:
# Set size limits
train_size_limit = 400
val_size_limit = 200

In [ ]:
# Create limited-size versions of train_full.jsonl and val_full.jsonl
with open(train_full_path, 'r') as train_full, open(content_dir / 'train.jsonl', 'w') as train_limit:
    for _ in range(train_size_limit):
        train_limit.write(next(train_full))

with open(val_full_path, 'r') as val_full, open(content_dir / 'val.jsonl', 'w') as val_limit:
    for _ in range(val_size_limit):
        val_limit.write(next(val_full))

In [2]:
import json
from huggingface_hub import snapshot_download
from pathlib import Path
import subprocess

/home/maia/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/requests/__init__.py:89: RequestsDependencyWarning: urllib3 (2.0.5) or chardet (3.0.4) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "


In [3]:
content_dir = Path('.').resolve()
original_config_path = content_dir / 'configs/rugpt35_13b.json'
model_dir = content_dir / "ruGPT-3.5-13B"
base_model = "ai-forever/ruGPT-3.5-13B"
output_dir = content_dir / 'output'
config_path = content_dir / 'configs/rugpt35_13b_colab.json'

In [4]:
# Paths to datasets
train_full_path = content_dir / 'train_full.jsonl'
train_small_path = content_dir / 'train.jsonl'
train_path = train_full_path  # change to train_full_path if you need
val_full_path = content_dir / 'val_full.jsonl'
val_small_path = content_dir / 'val.jsonl'
val_path = val_full_path  # change to val_full_path if you need

In [21]:
# Download binaries
snapshot_download(repo_id=base_model, local_dir=model_dir, ignore_patterns=["LICENSE", "README.md", ".gitattributes"])

Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 35.67it/s]


'/home/maia/maia/ruGPT-3.5-13B'

In [5]:
patch_model_config = True

if patch_model_config:
    replacements = {
        "tokenizer_config.json": {
            "add_bos_token": False,
            "add_prefix_space": False,
            "bos_token": {
                "__type": "AddedToken",
                "content": "<s>",
                "lstrip": False,
                "normalized": True,
                "rstrip": False,
                "single_word": False
            },
            "clean_up_tokenization_spaces": True,
            "eos_token": {
                "__type": "AddedToken",
                "content": "</s>",
                "lstrip": False,
                "normalized": True,
                "rstrip": False,
                "single_word": False
            },
            "errors": "replace",
            "mask_token": "<mask>",
            "model_max_length": 2048,
            "pad_token": {
                "__type": "AddedToken",
                "content": "<pad>",
                "lstrip": False,
                "normalized": True,
                "rstrip": False,
                "single_word": False
            },
            "tokenizer_class": "GPT2Tokenizer",
            "unk_token": {
                "__type": "AddedToken",
                "content": "<|endoftext|>",
                "lstrip": False,
                "normalized": True,
                "rstrip": False,
                "single_word": False
            },
            "padding_side": "left"
        },
        "special_tokens_map.json": {
            "bos_token": {
                "content": "<s>",
                "lstrip": False,
                "normalized": True,
                "rstrip": False,
                "single_word": False
            },
            "eos_token": {
                "content": "</s>",
                "lstrip": False,
                "normalized": True,
                "rstrip": False,
                "single_word": False
            },
            "mask_token": "<mask>",
            "pad_token": {
                "content": "<pad>",
                "lstrip": False,
                "normalized": True,
                "rstrip": False,
                "single_word": False
            },
            "sep_token": "<s>",
            "unk_token": {
                "content": "<|endoftext|>",
                "lstrip": False,
                "normalized": True,
                "rstrip": False,
                "single_word": False
            }
        },
        "generation_config.json": {
            "_from_model_config": True,
            "bos_token_id": 2,
            "eos_token_id": 3,
            "pad_token_id": 0,
            "temperature": 0.2,
            "top_p": 0.9,
            "top_k": 30,
            "do_sample": True,
            "max_new_tokens": 1536,
            "num_beams": 1,
            "repetition_penalty": 1.15,
            "no_repeat_ngram_size": 15
        },
    }

    print('Patching model config...')
    for filename, new_content in replacements.items():
        print(f'{filename}:')
        with (model_dir / filename).open() as fp:
            old_content = json.load(fp)
            print(f'    Original content: {old_content}')
            if old_content == new_content:
                print('    Already patched, skipping')
        print(f'    Updated content:  {new_content}')
        with (model_dir / filename).open('w') as fp:
            json.dump(new_content, fp, indent=4)

Patching model config...
tokenizer_config.json:
    Original content: {'add_bos_token': False, 'add_prefix_space': False, 'bos_token': {'__type': 'AddedToken', 'content': '<s>', 'lstrip': False, 'normalized': True, 'rstrip': False, 'single_word': False}, 'clean_up_tokenization_spaces': True, 'eos_token': {'__type': 'AddedToken', 'content': '</s>', 'lstrip': False, 'normalized': True, 'rstrip': False, 'single_word': False}, 'errors': 'replace', 'mask_token': '<mask>', 'model_max_length': 2048, 'pad_token': {'__type': 'AddedToken', 'content': '<pad>', 'lstrip': False, 'normalized': True, 'rstrip': False, 'single_word': False}, 'tokenizer_class': 'GPT2Tokenizer', 'unk_token': {'__type': 'AddedToken', 'content': '<|endoftext|>', 'lstrip': False, 'normalized': True, 'rstrip': False, 'single_word': False}, 'padding_side': 'left'}
    Already patched, skipping
    Updated content:  {'add_bos_token': False, 'add_prefix_space': False, 'bos_token': {'__type': 'AddedToken', 'content': '<s>', 'lst

In [28]:
# Load configurations
with original_config_path.open('r') as fp:
    config = json.load(fp)

# Colab adjustments
config['trainer']['per_device_train_batch_size'] = 2
config['trainer']['per_device_eval_batch_size'] = 1
config['trainer']['gradient_accumulation_steps'] = 128
config['trainer']['eval_steps'] = 50
config['trainer']['save_steps'] = 50
config['max_tokens_count'] = 1000
#config['model_name'] = str(model_dir)
config['templates_path'] = str(content_dir / 'internal_prompts/rugpt35.json')
config['load_in_8bit'] = True
config['load_in_4bit'] = False

In [30]:
# Demo adjustments
config['trainer']['eval_steps'] = 2
config['trainer']['logging_steps'] = 1
config['trainer']['num_train_epochs'] = 1
config['trainer']['bf16'] = False
config['trainer']['fp16'] = True

In [31]:
with config_path.open('w') as fp:
    json.dump(config, fp, indent=4)

# Run training
module_directory = Path('rulm/self_instruct').resolve()
subprocess.run(
    [
        'python3', '-m', 'src.train',
        '--config-file', config_path,
        '--train-file', train_path,
        '--val-file', val_path,
        '--output-dir', output_dir,
        '--report-to', 'none'
    ],
    cwd=module_directory,
    check=True
)

assert (output_dir / 'adapter_config.json').exists()

/usr/lib/python3/dist-packages/requests/__init__.py:89: RequestsDependencyWarning: urllib3 (2.0.5) or chardet (3.0.4) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "
Found safetensors installation, but --save_safetensors=False. Safetensors should be a preferred weights saving format due to security and performance reasons. If your model cannot be saved by safetensors please feel free to open an issue at https://github.com/huggingface/safetensors!
PyTorch: setting up devices
loading file vocab.json from cache at /home/maia/.cache/huggingface/hub/models--ai-forever--ruGPT-3.5-13B/snapshots/010145992ebc92d56cd969bc099b405d8ab82871/vocab.json
loading file merges.txt from cache at /home/maia/.cache/huggingface/hub/models--ai-forever--ruGPT-3.5-13B/snapshots/010145992ebc92d56cd969bc099b405d8ab82871/merges.txt
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at /home/maia/.cache/huggi

Vocab size:  50257
PAD:  0 <pad>
BOS:  2 <s>
EOS:  3 </s>
UNK:  1 <|endoftext|>
SEP:  None None
{'messages': [{'role': 'user', 'content': 'Я хотел бы узнать, можно ли редактировать документ в Word после его кодирования для повышения уникальности?'}, {'role': 'bot', 'content': 'Да, конечно, можно редактировать документ в Word после его кодирования. Тем не менее, следует отметить, что кодирование документа не гарантирует его полной уникальности. Редактирование документа после кодирования может увеличить его уникальность, однако это зависит от того, какие изменения были внесены и насколько они значительны.'}, {'role': 'user', 'content': 'Изменится только отредактированная часть или весь документ?'}, {'role': 'bot', 'content': 'Если вы редактируете только часть документа, то изменится только эта часть. Если же вы редактируете весь документ, то соответственно изменится и весь документ. Важно также понимать, что внесение изменений может повлиять на оригинальность всего документа, поэтому нео

  0%|          | 0/2958 [00:00<?, ?it/s]

[2, 88, 5102, 204, 3637, 577, 15559, 496, 17, 29527, 2246, 5856, 7208, 37632, 675, 19, 2168, 7271, 9975, 282, 5593, 288, 4733, 1115, 834, 19, 3, 204, 2, 40061, 204, 5371, 4926, 17, 1986, 34332, 405, 18, 491, 344, 11143, 1304, 1027, 1486, 7976, 7671, 3162, 36, 3, 204, 2, 71, 795, 204, 696, 1512, 19958, 18, 491, 344, 11143, 1304, 1027, 1486, 7976, 7671, 3162, 17, 1629, 383, 2709, 8542, 32477, 46802, 17991, 3171, 8504, 17, 984, 803, 2368, 1275, 19, 11694, 1388, 11102, 3192, 25309, 17, 383, 30993, 1027, 12295, 6099, 887, 288, 18153, 17, 288, 322, 3608, 13487, 11915, 1563, 33796, 17, 1894, 21814, 288, 2428, 26713, 1483, 1294, 19, 371, 3667, 17, 30993, 1027, 16390, 3383, 310, 2543, 5980, 352, 26440, 288, 322, 8435, 4818, 872, 19, 3, 204, 2, 40061, 204, 552, 435, 494, 9252, 282, 34332, 2309, 18, 491, 344, 11143, 5801, 36, 3, 204, 2, 71, 795, 204, 696, 1512, 19958, 18, 491, 344, 11143, 1304, 1027, 12295, 282, 15529, 2309, 1320, 5944, 7976, 5884, 5451, 19, 1919, 1027, 2543, 19398, 17, 984, 4272

100%|██████████| 2958/2958 [00:13<00:00, 218.35it/s]


...

{'loss': 1.6539, 'learning_rate': 0.0002909538931178862, 'epoch': 0.24}



100%|█████████▉| 2957/2958 [30:05<00:00,  1.56it/s]
                                                        
100%|██████████| 2958/2958 [30:05<00:00,  1.60it/s]
                                                   

{'eval_loss': 1.6216771602630615, 'eval_runtime': 1806.4901, 'eval_samples_per_second': 1.637, 'eval_steps_per_second': 1.637, 'epoch': 0.24}


 24%|██▍       | 53/219 [18:11:35<64:53:57, 1407.45s/it]

{'loss': 1.6837, 'learning_rate': 0.0002900816920379045, 'epoch': 0.24}


In [6]:
# Fix config of trained model
with (output_dir / 'generation_config.json').open('w') as fp:
    json.dump({
        "bos_token_id": 2,
        "eos_token_id": 3,
        "pad_token_id": 0,
        "temperature": 0.2,
        "top_p": 0.9,
        "top_k": 30,
        "do_sample": True,
        "max_new_tokens": 1536,
        "num_beams": 1,
        "repetition_penalty": 1.15,
        "no_repeat_ngram_size": 15,
    }, fp, indent=4)

In [1]:
from llm_rs.convert import AutoConverter
from llm_rs import AutoQuantizer, QuantizationType, ContainerType
from pathlib import Path

/home/maia/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/requests/__init__.py:89: RequestsDependencyWarning: urllib3 (2.0.5) or chardet (3.0.4) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "


In [2]:
content_dir = Path('.').resolve()
ggml_f16_model_name = 'ruGPT-3.5-13B-lora-f16.bin'
input_ggml = content_dir / 'ruGPT-3.5-13B-lora'
output_ggml = content_dir / 'output_ggml'

In [3]:
converted_model = AutoConverter.convert(input_ggml, output_ggml)

assert (output_ggml / ggml_f16_model_name).exists()

Loading checkpoint shards: 100%|██████████| 6/6 [19:37<00:00, 196.30s/it]
/home/maia/.local/lib/python3.8/site-packages/peft/tuners/lora.py:475: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [4]:
# Quantize the model to different formats
AutoQuantizer.quantize(converted_model, quantization=QuantizationType.Q4_1, container=ContainerType.GGML)

'/home/maia/maia/output_ggml/ruGPT-3.5-13B-lora-q4_1.bin'

In [5]:
AutoQuantizer.quantize(converted_model, quantization=QuantizationType.Q5_0, container=ContainerType.GGML)

'/home/maia/maia/output_ggml/ruGPT-3.5-13B-lora-q5_0.bin'

In [ ]:
AutoQuantizer.quantize(converted_model, quantization=QuantizationType.Q5_1, container=ContainerType.GGML)

In [ ]:
AutoQuantizer.quantize(converted_model, quantization=QuantizationType.Q8_0, container=ContainerType.GGML)